# Анализ лояльности пользователей Яндекс Афиши

# Цели проекта
* Провести исследовательский анализ лояльности пользователей Яндекс Афиши для выявления факторов, влияющих на лояльность пользователей.

# Задачи
* Загрузить данные и провести их первичный осмотр.
* Провести предобработку данных.
* Сформировать профиль пользователей 
* Провести исследовательский анализ данных:
    * Исследовать признакаки первого заказа и их связи с лояльностью клиентов
    * Проанализировать возвраты пользователей
    * Исследовать поведение пользователей через показатели выручки и состава заказа
    * Исследовать влияние временных характеристик первого заказа на повторные покупки
    * Провести корреляционный анализ количества покупок и признаков пользователя
* Сформулировать выводы и практические рекомендации для команды маркетинга по повышению лояльности пользователей

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Используйте следующие параметры для подключения к базе данных `data-analyst-afisha`:

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


In [ ]:
# Импортируем все необходимые библиотеки
!pip install sqlalchemy psycopg2-binary python-dotenv phik
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import phik
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(override=True)

# Формируем строку подключения
db_config = {
    'user': os.getenv('DB_USER') or 'data-analyst-afisha',
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST') or 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': os.getenv('DB_PORT') or '6432',
    'db': os.getenv('DB_NAME')
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

# Создаем движок с SSL
engine = create_engine(
    connection_string,
    connect_args={'sslmode': 'require'})

In [ ]:
# Задаем запрос
query = '''
-- Настройка параметра synchronize_seqscans важна для проверки
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)

  SELECT user_id,
    p.device_type_canonical,
    p.order_id,
    p.created_dt_msk AS order_dt,
    p.created_ts_msk AS order_ts,
    p.currency_code,
    p.revenue,
    p.tickets_count,
    EXTRACT(DAY FROM (created_dt_msk - LAG(created_dt_msk) OVER (PARTITION BY user_id ORDER BY created_dt_msk ASC))) AS days_since_prev,
    p.event_id,
    e.event_name_code AS event_name,
    e.event_type_main,
    p.service_name,
    r.region_name,
    c.city_name
  FROM purchases AS p
  JOIN events AS e ON p.event_id = e.event_id
  JOIN city AS c ON e.city_id = c.city_id
  JOIN regions AS r ON c.region_id = r.region_id
  WHERE p.device_type_canonical IN ('mobile', 'desktop') AND e.event_type_main != 'фильм'
  ORDER BY user_id
'''
# Вывод результата
df = pd.read_sql_query(query, con=engine)
display(df.head())

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
# Выводим информацию о датафрейме
df.info()

В датафрейме 14 столбцов и 290611 строк.
Пропуски присутствуют только в столбце `days_since_prev` и связаны с особенностями данных

Типа данных:
* 7 столбцов: `user_id`, `device_type_canonical`, `currency_code`, `event_type_main`, `service_name`, `region_name` и `city_name` принадлежат к типу `object`, что соответсвует данным
* 3 столбца: `order_id`, `tickets_count` и `event_id` принадлежат к типу `int64`, что соответствует данным 
* 2 столбца: `revenue` и `days_since_prev` принадлежат к типу `float64`. Столбец `days_since_prev` может быть приведен к типу `integer`, так как содержит данные о целочисленном кол-ве дней.
* 2 столбца: `order_dt` и `order_ts` принадлежат к типу `datetime64`, что соответствует данным

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [ ]:
# Загружаем датасет с информацией о курсе казахстанского тенге за 2024 год
df_kzt = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

In [ ]:
# Выводим информацию о датасете и первые строки датасета для ознакомления
df_kzt.info()
display(df_kzt.head())

In [ ]:
# Приводим столбец date к типу datetime
df_kzt['data']=df_kzt['data'].astype('datetime64[ns]')

In [ ]:
# Выводим информацию для проверки
df_kzt.info()

In [ ]:
# Объединяем датафреймы
df=df.merge(df_kzt[['data', 'curs']], left_on='order_dt', right_on='data', how='left')

In [ ]:
# Расчитываем столбец revenue_rub
df['revenue_rub'] = df['revenue']
df.loc[(df['currency_code'] == 'kzt'), 'revenue_rub'] = (df.loc[(df['currency_code'] == 'kzt'), 'revenue'] 
                                                         * df.loc[(df['currency_code'] == 'kzt'), 'curs']/100) 

In [ ]:
# Удаляем лишний столбец data
df.drop(columns='data', inplace=True)

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

Так как в датафрейме отсутствуют пропуски, кроме столбца days_since_prev и связаны с особенностями данных, то переходим к поискам дубликатов

In [ ]:
# Изначальное кол-во строк датафрейма
initial_row_count = df.shape[0]
display(f'Изначальное количество строк датафрейма: {initial_row_count}')

In [ ]:
# Определяем явные дубликаты
display(f'Количество явных дубликатов: {df.duplicated().sum()}')

In [ ]:
# Определяем дубликаты по столбцам
df[['user_id', 'event_id', 'order_ts', 'revenue', 'tickets_count', 'currency_code']].duplicated().sum()
dupl_row_count = df[['user_id', 'event_id', 'order_ts', 'revenue', 'tickets_count', 'currency_code']].duplicated().sum()
display(f'Количество дубликатов {dupl_row_count}')

In [ ]:
# Удаляем дубликаты 
df = df.drop_duplicates(subset=['user_id', 'event_id', 'order_ts', 'revenue', 'tickets_count', 'currency_code'], keep='first')

In [ ]:
# Кол-во строк датафрейма после удаления дубликатов
without_dupl_row_count = df.shape[0]
display(f'Количество строк датафрейма после удаления дубликатов: {without_dupl_row_count}')
display(f'Доля удаленных строк: {round(dupl_row_count/initial_row_count*100, 3)}%')

In [ ]:
# Выводим количество уникальных значений 
for column in df[['device_type_canonical', 'currency_code', 'event_type_main', 'service_name', 'region_name', 'city_name']]:
    display(f"Количество уникальные значения столбца {column}: {df[column].nunique()}")

In [ ]:
# Проведим нормализацию
df_norm = df
for column in df_norm[['device_type_canonical', 'currency_code', 'event_type_main', 'service_name', 'region_name', 'city_name']]:
    df[column].str.lower().str.strip()

In [ ]:
# Выводим количество уникальных значений для проверки 
for column in df_norm[['device_type_canonical', 'currency_code', 'event_type_main', 'service_name', 'region_name', 'city_name']]:
    display(f"Количество уникальные значения столбца {column}: {df[column].nunique()}")

Так как кол-во уникальных значений осталось неизменный, заключаем, что неявные дубликаты отсутствуют

In [ ]:
# Выводим статистические признаки численных стобцов датафрейма
df[['tickets_count', 'days_since_prev', 'revenue_rub']].describe(percentiles=[0.25, 0.50, 0.75, 0.9, 0.99])

In [ ]:
# Построим график распределения для столбца revenue_rub
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='revenue_rub', color='blue')

# Задаем настройки графика
plt.title('График распределения выручки')
plt.xlabel('Выручка')
plt.ylabel('Количество')


In [ ]:
# Построим диаграмму размаха для столбца revenue_rub
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='revenue_rub', color='blue')

# Задаем настройки графика
plt.title('Диаграмма размаха выручки')
plt.xlabel('Выручка')

* Столбец `tickets_count` содержит максимальное значение, равное 57, можно привести к типу `int8` для оптимизации.
* Столбец `days_since_prev` - приводим к целочисленному типу данных `int16`.
* В столбце `revenue_rub` наблюдаем наличие отрицательных значений, медиана превышает среднее значение, что указывает на скос данных влево, так как максимальное значение значительно превышает значение 99-го перцентиля, заключаем, что имеются выбросы. 

In [ ]:
# Находим значение 99-го процентиля для столбца revenue_rub
revenue_limit = df['revenue_rub'].quantile(0.99)

In [ ]:
# Фильтруем столбец revenue_rub по 99-му процентилю и убираем отрицательные значения
df = df.loc[(df['revenue_rub'] <= revenue_limit) & (df['revenue_rub'] >= 0)]

In [ ]:
# Распределение количества билетов 
df_tickets = df.groupby('tickets_count')['order_id'].count()
display(df_tickets)

In [ ]:
# Отсекаем выбросы в столбце tickets_count по 99-му процентилю
tickets_limit = df['tickets_count'].quantile(0.99)
df = df.loc[(df['tickets_count'] <= tickets_limit)]

In [ ]:
# Распределение количества билетов без выбросов
df_tickets_sort = df.groupby('tickets_count')['order_id'].count()
display(f'Распределение кол-ва билетов в абсолютных значениях')
display(df_tickets_sort)

# Распределение количества билетов без выбросов в процентном отношении 
df_tickets_sort_perc = round(df_tickets_sort/df_tickets_sort.sum(), 2)
display(f'Распределение кол-ва билетов в процентном отношении')
display(df_tickets_sort_perc)

In [ ]:
# Строим график распределения столбца tickets_count
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='tickets_count', color='blue', discrete=True)

# Задаем настройки графика
plt.title('График распределения количества билетов без выбросов')
plt.xlabel('Количество билетов')
plt.ylabel('Количество')

Основная масса заказов состоит из 2-3 билетов, после 4 билетов количество покупок снижается. В основном покупки совершаются небольшими компаниями.

In [ ]:
# Выводим статистические признаки после фильтрации данных
df[['tickets_count', 'days_since_prev', 'revenue_rub']].describe(percentiles=[0.25, 0.50, 0.75, 0.9, 0.99])

In [ ]:
# Создаем копию датафрейма
df_filtered = df.copy()

In [ ]:
# Приводим столбцец tickets_count к меньшей размерности 
df_filtered['tickets_count'] = pd.to_numeric(df_filtered['tickets_count'], downcast='integer')

In [ ]:
# Приводи столбцец days_since_prev к типу int16
df_filtered['days_since_prev'] = df_filtered['days_since_prev'].astype('Int16')

In [ ]:
# Проверяем типы данных
df_filtered[['tickets_count', 'days_since_prev']].info()

In [ ]:
# Определяем кол-во отсеченных данных
filtered_df_row_count = df_filtered.shape[0]
display(f'Количество строк датафрейма после фильтрации: {filtered_df_row_count}')
display(f'Количество отсеченных строк: {without_dupl_row_count - filtered_df_row_count}')
display(f'Доля отсеченных строк: {round((1-filtered_df_row_count/without_dupl_row_count)*100, 3)}%')
display(f'Общая доля отсеченных строк : {round((1-filtered_df_row_count/initial_row_count)*100, 3)}%')

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


In [ ]:
# Строим профиль пользователя и добавляем два признака совершил ли пользователь 2 или 5 и более заказов:
df_user_pr = (df_filtered.sort_values(by='order_ts').groupby('user_id').agg(
                                                                    first_order=('order_dt', 'min'),
                                                                    last_order=('order_dt', 'max'),
                                                                    device=('device_type_canonical', 'first'),
                                                                    region=('region_name', 'first'),
                                                                    service=('service_name', 'first'),
                                                                    first_event_type=('event_type_main', 'first'),
                                                                    orders_count=('order_id', 'count'),
                                                                    avg_revenue=('revenue_rub', 'mean'),
                                                                    avg_tickets_count=('tickets_count', 'mean'),
                                                                    avg_days_since_prev=('days_since_prev', 'mean'))
                                                                .assign(
                                                                    is_two = lambda x: x['orders_count'] >= 2,
                                                                    is_five = lambda x: x['orders_count'] >=5).reset_index())

# Определим кол-во строк полученного датафрейма
row_count = df_user_pr.shape[0]

# Выводим первые строки для ознакомления 
df_user_pr.head()

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
# Общее количество пользователей 
user_count = df_user_pr.shape[0]
display(f'Общее число пользователей: {user_count}')

In [ ]:
# Рассчитываем среднюю выручку с одного заказа
order_avg_revenue = df_user_pr['avg_revenue'].mean().round(2)
display(f'Средняя выручка с одного заказа: {order_avg_revenue}')

In [ ]:
# Рассчитываем долю пользователей, совершивших 2 и более заказа
is_two_share = round(df_user_pr['is_two'].sum()/df_user_pr.shape[0]*100, 2)
display(f'Доля пользователей, совершивших 2 и более заказов: {is_two_share}%')

In [ ]:
# Рассчитываем долю пользователей, совершивших 5 и более заказа
is_five_share = round(df_user_pr['is_five'].sum()/df_user_pr.shape[0]*100, 2)
display(f'Доля пользователей, совершивших 2 и более заказов: {is_five_share}%')

In [ ]:
# Выводим статистические показатели по столбцам: orders_count, avg_tickets_count, avg_days_since_prev
df_user_pr[['orders_count', 'avg_tickets_count', 'avg_days_since_prev']].describe(percentiles=(0.25, 0.50, 0.75, 0.95, 0.99))

In [ ]:
# Строим диаграмму размаха для столбца orders_count
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_user_pr, x='orders_count', color='blue')

# Задаем настройки графика
plt.title('Диаграмма размаха количества заказов')
plt.xlabel('Количество заказов')

In [ ]:
# Строим диаграмму размаха для столбца avg_tickets_count
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_user_pr, x='avg_tickets_count', color='blue')

# Задаем настройки графика
plt.title('Диаграмма размаха среднего количества билетов')
plt.xlabel('Количество билетов')

In [ ]:
# Строим диаграмму размаха для столбца avg_days_since_prev
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_user_pr, x='avg_days_since_prev', color='blue')

# Задаем настройки графика
plt.title('Диаграмма размаха среднего количества дней между покупками')
plt.xlabel('Количество дней')

* В столбце `orders_count` обнаружены аномальные значения. Максимальное значение равно 10155, когда 99-ый процентиль равен 151.
Проведем фильтрацию данных по 99-му процентилю
* В столбцах `avg_tickets_count` и `avg_days_since_prev` не было обнаружено аномальных данных

In [ ]:
# Определяем 99-ый процентиль столбца orders_count
orders_count_limit = df_user_pr['orders_count'].quantile(0.99)

# Проводим фильтрацию данных 
df_user_pr = df_user_pr.loc[(df_user_pr['orders_count'] <= orders_count_limit)]

In [ ]:
# Определяем кол-во строк после фильтрации
filtered_row_count = df_user_pr.shape[0]
display(f'Количество строк после фильтрации: {filtered_row_count}')
display(f'Количество отсеченных строк: {row_count - filtered_row_count}')
display(f'Доля отсеченных строк: {round((1-filtered_row_count/row_count)*100, 3)}%')

In [ ]:
# Выводим статистические показатели после фильрации данных
df_user_pr[['orders_count', 'avg_tickets_count', 'avg_days_since_prev']].describe(percentiles=(0.25, 0.5, 0.75, 0.9, 0.99))

* Были сформированы профили пользователей
* Добавлены бинарные признаки: совершено 2 и более заказов, совершено 5 и более заказов
* Рассчитаны общие покказатели: 
    * общее число пользователей - 21831 пользователь
    * средняя выручка с одного заказа - 544,79 рублей
    * доля пользователей, совершивших 2 и более заказа - 61.70%
    * доля пользователей, совершивших 5 и более заказов - 28.99%
* Была проведена фильтрация данных по 99-му процентилю кол-ва заказов для отсечения выбросов, отсечено 219 строк

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


In [ ]:
# Проводим группировку пользователей по типу их первого мероприятия в абсолютных значениях
user_first_event = df_user_pr.groupby('first_event_type')['user_id'].count().sort_values(ascending=False)

# Переводим в процентное отношение
user_first_event_perc = round(user_first_event/user_first_event.sum()*100, 2)

display(f'Распределение пользователей по типу первого мероприятия в абсолютных значениях')
display(user_first_event)
display(f'Распределение пользователей по типу первого мероприятия в процентном отношении')
display(user_first_event_perc)

In [ ]:
# Строим график распределения пользователей по типу их первого мероприятия
user_first_event.plot.bar(title='Распределение пользователей по типу их первого мероприятия', figsize=(12, 8), rot=0)

# Задаем настройки графика
plt.xlabel('Тип мероприятия')
plt.ylabel('Количество пользователей')
plt.grid(axis='y')

По распределению пользователей видно:
* самым популярным выбором в качестве первого мероприятия являются концерты
* вторым по популярности выбором являются театры
* категорию "другое" не учитываем, так как она включает в себя: события, спектакли, выставки, цирковые шоу и т.п. 

In [ ]:
# Проводим группировку пользователей по типу устройства, с которого совершена первая покупка
user_first_device = df_user_pr.groupby('device')['user_id'].count().sort_values(ascending=False)

# Приводим к процентному отношению
user_first_device_perc = round(user_first_device/user_first_device.sum()*100, 2)

display(f'Распределение пользователей по типу устройства, с которого совершена первая покупка, в абсолютных значениях')
display(user_first_device)
display(f'Распределение пользователей по типу устройства, с которого совершена первая покупка, в процентном отношении')
display(user_first_device_perc)

In [ ]:
# Строим график распределения пользователей по типу устройства, с которого совершена первая покупка
user_first_device.plot.pie(title='Распределение пользователей по типу устройства, с которого совершена первая покупка', 
                               autopct='%1.2f%%', figsize=(12, 6))

# Задаем настройки графика
plt.xlabel('Тип устройства')
plt.ylabel('Доля пользователей')

* Из графика распределения видно, что первую покупку преимущественно совершают с мобильного телефона

In [ ]:
# Проводим группировку пользователей по региону проведения мероприятия из первого заказа
user_first_region = df_user_pr.groupby('region')['user_id'].count().sort_values(ascending=False).head(10)

# Приводим к процентному отношению
user_first_region_perc = round(user_first_region/user_first_region.sum()*100, 2)

display(f'Распределение пользователей по региону проведения мероприятия из первого заказа в абсолютных значениях')
display(user_first_region)
display(f'Распределение пользователей по региону проведения мероприятия из первого заказа в процентном отношении')
display(user_first_region_perc)

Выборка данных включает в себя 81 регион, так как оснавная масса пользователей распределена в нескольких регионах, выводим распределение только для 10 регионов 

In [ ]:
# Строим график распределения пользователей по региону проведения мероприятия из первого заказа
user_first_region.plot.bar(title='Распределение пользователей по региону проведения мероприятия из первого заказа',
                          figsize=(12, 8), rot=45)

# Задаем настройки графика
plt.xlabel('Регион')
plt.ylabel('Количество пользователей')
plt.grid(axis='y')

* Из графика распределения видно, что преимущественно первые заказы были совершены в Камневском регионе и Североярской области

In [ ]:
# Проводим группировку пользователей по билетному оператору, продавшему билеты на первый заказ
user_first_service = df_user_pr.groupby('service')['user_id'].count().sort_values(ascending=False).head(10)

# Приводим к процентному отношению
user_first_service_perc = round(user_first_service/user_first_service.sum()*100, 2)

display(f'Распределение пользователей по билетному оператору, продавшему билеты на первый заказ, в абсолютных значениях')
display(user_first_service)
display(f'Распределение пользователей по билетному оператору, продавшему билеты на первый заказ, в процентном отношении')
display(user_first_service_perc)

Выборка данных включает в себя 34 билетных опереаторов, так как оснавная масса пользователей покупает у определенных операторов, выводим распределение только для 10 операторов 

In [ ]:
# Строим график распределения пользователей по билетному оператору, продавшему билеты на первый заказ
user_first_service.plot.bar(title='Распределение пользователей по билетному оператору, продавшему билеты на первый заказ',
                          figsize=(12, 8), rot=45)

# Задаем настройки графика
plt.xlabel('Билетный оператор')
plt.ylabel('Количество пользователей')
plt.grid(axis='y')

* Из графика распределения видно, что билетный оператор "Билеты без проблем" является самым популярный среди пользователей

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
# Проводим группировку пользователей, совершивших 2 и более покупок, в сегменте типа мероприятия
user_event_is_two = (df_user_pr.query('is_two == True')
                    .groupby('first_event_type')['user_id'].count().sort_values(ascending=False))

# Выводим доли пользователей, совершивших 2 и более покупок, в сегменте типа мероприятия
user_event_share = round(user_event_is_two/user_first_event*100, 2).sort_values(ascending=False)

# Определим среднюю долю пользователей, совершивших 2 и более покупок, по всем типам мероприятий
mean_share = user_event_share.mean().round(2)

display(f'Доли пользователей, совершивших 2 и более покупок, в сегменте типа мероприятий')
display(user_event_share)

In [ ]:
# Строим график распределение долей пользователей совершивших 2 или более покупок,
#                                                                   относительно общего кол-ва пользователей в каждом сегменте
user_event_share.plot.bar(
    title='Доли пользователей, совершивших 2 или более покупок, относительно общего кол-ва пользователей в каждом сегменте',
    rot=0, figsize=(12, 8))

# Отображаем на графике среднюю долю пользователей по всем типам заведения 
plt.axhline(mean_share, color='red', linestyle='--')

# Задаем настройки графика
plt.xlabel('Доля в процентах')
plt.ylabel('Тип мероприятия')
plt.grid(axis='y')

* Из графика видно, что в целом доля пользователей, совершивших 2 или более покупок, незначительно разнится от типа заведения.
* Однако выставки, театры и концерты имеют незначительно превышают среднюю долю пользователей по всем типам заведений.

In [ ]:
# Проводим группировку пользователей, совершивших 2 и более покупок, в сегменте типа устройства
user_device_is_two = (df_user_pr.query('is_two == True')
                    .groupby('device')['user_id'].count().sort_values(ascending=False))

# Выводим долю пользователей, совершивших 2 и более покупок, в сегменте типа устройства
user_device_share = round(user_device_is_two/user_first_device*100, 2).sort_values(ascending=False)

display(f'Доли пользователей, совершивших 2 и более покупок, в сегменте типа устройства')
display(user_device_share)

In [ ]:
# Строим график распределения долей пользователей по типу устройства, с которого совершена первая покупка
user_device_share.plot.bar(title='Доли пользователей по типу устройства, совершивших 2 и более покупок', 
                           rot=0, figsize=(12, 6))

# Задаем настройки графика
plt.xlabel('Тип устройства')
plt.ylabel('Доля пользователей')
plt.grid(axis='y')

* Из графика видно, что доли пользователей, совершивших 2 и более покупок, незначительно отличаются от типа устройства

In [ ]:
# Проводим группировку пользователей, совершивших 2 и более покупок, в сегменте региона
user_region_is_two = (df_user_pr.query('is_two == True')
                    .groupby('region')['user_id'].count().sort_values(ascending=False)).head(10)

# Выводим долю пользователей, совершивших 2 и более покупок, в сегменте региона
user_region_share = round(user_region_is_two/user_first_region*100, 2).sort_values(ascending=False)

# Определяем среднюю долю пользователей, совершивших 2 и более покупок, по всем регионам
mean_region_share = user_region_share.mean().round(2)

display(f'Доли пользователей, совершивших 2 и более покупок, в сегменте региона')
display(user_region_share)

In [ ]:
# Строим график распределения долей пользователей по регионам
user_region_share.plot.bar(title='Доли пользователей по регионам, совершивших 2 и более покупок', 
                           rot=45, figsize=(12, 6))

# Отображаем на графике среднюю долю пользователей по всем регионам
plt.axhline(mean_region_share, color='red', linestyle='--')

# Задаем настройки графика
plt.xlabel('Регион')
plt.ylabel('Доля пользователей')
plt.grid(axis='y')

* Из графика видно, что доли пользователей, совершивших 2 или более покупок, также незначительно разнятся от региона.
* Однако доли пользователей в Шанырском регионе, Светополянском округе, Широковской области и Североярской области незначительно превышают среднюю долю пользователей по всем регионам 

In [ ]:
# Проводим группировку пользователей, совершивших 2 и более покупок, в сегменте билетного оператора
user_service_is_two = (df_user_pr.query('is_two == True')
                    .groupby('service')['user_id'].count().sort_values(ascending=False)).head(10)

# Выводим долю пользователей, совершивших 2 и более покупок, в сегменте билетного оператора
user_service_share = round(user_service_is_two/user_first_service*100, 2).sort_values(ascending=False)

# Определяем среднюю долю пользователей, совершивших 2 и более покупок, по всем регионам
mean_service_share = user_service_share.mean().round(2)

display(f'Доли пользователей, совершивших 2 и более покупок, в сегменте билетного оператора')
display(user_service_share)

In [ ]:
# Строим график распределения долей пользователей, совершивших 2 и более покупок, в сегменте билетного оператора
user_service_share.plot.bar(title='Доли пользователей по билетным операторам, совершивших 2 и более покупок', 
                           rot=45, figsize=(12, 6))

# Отображаем на графике среднюю долю пользователей по всем регионам
plt.axhline(mean_service_share, color='red', linestyle='--')

# Задаем настройки графика
plt.xlabel('Билетный оператор')
plt.ylabel('Доля пользователей')
plt.grid(axis='y')

* Из графика видно, что доли пользователей, совершивших 2 или более покупок, также незначительно разнятся от билетного оператора

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

* Гипотеза № 1 не подтвердилась. Тип мероприятия незначительно влияет на вероятность возврата. Предположение, что пользователи, совершившие первый заказ на спортивные мероприятия, совершают повторные заказы чаще, чем пользователи, оформившие первый заказ на концерты, оказалось неверным. Не смотря на незначительные отличия по всем типам мероприятий, концерты показали чуть большую долю пользователей, совершивших 2 или более заказов, чем спортивные мероприятия. 

* Гипотеза № 2 также не подтвердилась. Предположение, что регионы с наибольшим количеством посещений имеют большую долю повторных заказов, чем менее активные регионы, оказалось неверным. Доли повторных заказов лишь незначительно зависят от активности регионов. Так Каменевский регион имеет наибольшую активность, однако доля повторных заказов близка к средней доли повторных заказов. А Шанырский регион имеет низкую активность, однако доля повторных заказов незначительно превышает среднюю долю повторных заказов по всем регионам

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


In [ ]:
# Строим график распределения средней выручки для пользователей совершивших один заказ и вернувшихся пользователей
plt.figure(figsize=(14, 8))
sns.histplot(data=df_user_pr, x='avg_revenue', hue='is_two', bins = 50, kde=True)

# Задаем настройки графика
plt.title('График распределения средней выручки для пользователей совершивших один заказ и вернувшихся пользователей')
plt.xlabel('Средняя выручка')
plt.ylabel('Количество пользователей')
plt.grid(axis='both')

In [ ]:
# Выводим статистические показатели для пользователей совершивших один заказ
one_order = df_user_pr.query('is_two == False')['avg_revenue'].describe()

display(f'Статистические показатели по средней выручке для пользователей, совершивших один заказ')
display(one_order)

In [ ]:
# Выводим статистические показатели для пользователей совершивших два и более заказов
two_order = df_user_pr.query('is_two == True')['avg_revenue'].describe()

display(f'Статистические показатели по средней выручке для пользователей, совершивших два и более заказов')
display(two_order)

* Из распределения средней выручки для пользователей, совершивших один заказ, видно:
    * среднее значение равно 545 рублям, медианное значение равно 378 рублям, стандартное отклонение равно 519 рублей, что говорит о большой вариативности данных. Основной диапозон средней выручки от 0 до 250 рублей
    * большое количество пользователей с чеком в 0 рублей может говорить о посещении бесплатных мероприятий и отсутствия интереса к другим типам мероприятий
* Из распределения средней выручки для пользователей, совершивших два заказа и более, видно: 
    * среднее значение равно 544 рублям, медианное значение равно 497 рублям, стандартное отклонение равно 399 рублей, что говорит об умеренной вариативности данных. Основной диапозон средней выручки от 300 до 800 рублей 
    * более высокое медианное значение может указывать на стабильный интерес покупателей к мероприятиям

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
# Проводим группировку по пользователям, совершивших 2-4 заказа
two_four_group = df_user_pr.query("is_two == True & is_five == False")

# Проводим группировку по пользователям, совершивших 5 и более заказов
five_group = df_user_pr.query("is_five == True")

In [ ]:
# Строим график распределения средней выручки для пользователей, совершивших 2-4 заказа и совершивших 5 и более заказов
plt.figure(figsize=(14, 8))
sns.histplot(data=two_four_group, x='avg_revenue', bins = 50, color='red', alpha=0.5, kde=True,
             label='Пользователи, совершивших 2-4 заказа')

# Строим график распределения средней выручки для пользователей, совершивших 5 и более заказов
sns.histplot(data=five_group, x='avg_revenue', bins = 50, alpha=0.5, kde=True,
             label='Пользователи, совершивших 5 и более заказов')

# Задаем настройки графика
plt.title('Распределение среднего чека от группы пользователей')
plt.xlabel('Средняя выручка')
plt.ylabel('Количество пользователей')
plt.legend(title='Группы пользователей')
plt.grid(axis='both')

In [ ]:
# Выводим статистические показатели по средней выручки пользователей, совершивших 2-4 заказа
display(f'Статистические показатели по средней выручке для пользователей, совершивших 2-4 заказа')
display(two_four_group['avg_revenue'].describe())

In [ ]:
# Выводим статистические показатели по средней выручки пользователей, совершивших 5 и более заказов
display(f'Статистические показатели по средней выручке для пользователей, совершивших 5 и более заказов')
display(five_group['avg_revenue'].describe())

* Из распределения средней выручки для пользователей, совершивших 2-4 заказа, видно:
    * среднее значение равно 551 рублям, медианное значение равно 472 рублям, стандартное отклонение равно 420 рублей, что говорит об умеренной вариативности данных. Основной диапозон средней выручки от 0 до 350 рублей
    * данные имеют сильный скос влево
* Из распределения средней выручки для пользователей, совершивших 5 и более заказов, видно: 
    * среднее значение равно 536 рублям, медианное значение равно 512 рублям, стандартное отклонение равно 298 рублей, что говорит о более компктном распределении данных. Основной диапозон средней выручки от 350 до 800 рублей.
    * данные ближе к нормальному распределению относительно группы с 2-4 заказами (данные смещены правее) 

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
# Строим график распределения пользователей по среднему количеству билетов в заказе
plt.figure(figsize=(12,8))
sns.histplot(data=df_user_pr, x='avg_tickets_count', bins=15)

# Задаем настройки графика
plt.title('Распределение пользователей по среднему количеству билетов')
plt.xlabel('Среднее количество билетов в заказе')
plt.ylabel('Количество пользователей')
plt.grid(axis='both')

In [ ]:
# Выводим статистические показатели по среднему количеству билетов в заказе
display(f'Статистические показатели по среднему количеству билетов в заказе')
display(df_user_pr['avg_tickets_count'].describe(percentiles=(0.25, 0.5, 0.75, 0.9)))

* Из распределения видно, что основная масса пользователей берет по 2-3 билета

In [ ]:
# Разбиваем пользователей на сегменты по среднему количеству билетов в заказе
df_user_pr['ticket_segments'] = pd.cut(df_user_pr['avg_tickets_count'],
                                      bins=[1, 2, 3, 5, float('inf')],
                                      labels=['1-2 билета', '2-3 билета', '3-5 билетов', '5 и более билетов'],
                                      include_lowest=True, right=False)

In [ ]:
# Рассчитываем общее количество пользователей и доли вернувшийся пользователей в каждом сегменте 
tickets = df_user_pr.groupby('ticket_segments').agg(
                                                    count=('user_id', 'count'),
                                                    share_is_two=('is_two', 'mean')).reset_index().round(2)

display(f'Общее количество пользователей и доли вернувшихся покупателей в каждом сегменте')
display(tickets)

In [ ]:
# Создаем сетку для графиков
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 8))

# Строим графики распределения пользователей по сегментам
tickets.plot.bar(x='ticket_segments', y='count', ax=axes[0], rot= 45, legend=False)
# Задаем настройки графика распределения пользователей по сегментам
axes[0].set_title('Количество пользователей в сегментах')
axes[0].set_xlabel('Сегменты')
axes[0].set_ylabel('Количество покупателей')

# Строим графики распределения долей вернувшихся покупателей по сегментам
tickets.plot.bar(x='ticket_segments', y='share_is_two', ax=axes[1], rot=45, legend=False)
# Задаем настройки графика распределения долей вернувшихся пользователей по сегментам
axes[1].set_title('Доля вернувшихся пользователей в сегментах')
axes[1].set_xlabel('Сегменты')
axes[1].set_ylabel('Доля вернувшихся покупателей')

* Из распределения пользователей по среднему количеству билетов в заказе видно:
    * распределение по сегментам неравномерное, основная масса покупок приходится на 2-3 билета и на 3-5 билетов 
    * наибольшое количество пользователей и наибольшая доля вернувшихся пользователей лежат в сегменте 2-3 билетов
    * сегменты 3-5 билетов и 1-2 билета имеют незначительное отличие в доли вернувшихся пользователей
    * сегмент с 5 и более билетами имеет самые низкие показатели

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
# Определяем общее количество покупателей и долю вернувшихся покупателей по дню совершения их первого заказа
first_order = df_user_pr.groupby(df_user_pr['first_order'].dt.day_name()).agg(
                                                                              count=('user_id', 'count'),
                                                                              share=('is_two', 'mean')).round(3)

# Задаем новый порядок индексов
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
first_order = first_order.reindex(days) 

mean_count_days = first_order['count'].mean()
mean_share_days = first_order['share'].mean()

display('Распределение общего количества пользователей и долей вернувшихся пользователей в разрезе дней недели')
display(first_order)

In [ ]:
# Создаем сетку для графиков
fig,axes = plt.subplots(nrows=1, ncols=2, figsize=(12,8))

# Строим график распределения количества пользователей по дням недели их первого заказа
first_order.plot.bar(y='count', ax=axes[0], legend=False, rot=45)
# Задаем настройки графика и отображаем линией среднее кол-во пользователей по всем дням
axes[0].set_title('Распределение количества пользователей')
axes[0].set_xlabel('День недели')
axes[0].set_ylabel('Количество пользователей')
axes[0].axhline(mean_count_days, color='red', linestyle='--')

# Строим график распределения долей вернувшихся пользователей по дням недели их первого заказа
first_order.plot.bar(y='share', ax=axes[1], legend=False, rot=45)
# Задаем настройки графика и отображаем линией среднюю долю вернувшийхся пользователей по всем дням
axes[1].set_title('Распределение долей вернувшихся пользователей')
axes[1].set_xlabel('День недели')
axes[1].set_ylabel('Доля пользователей')
axes[1].axhline(mean_share_days, color='red', linestyle='--')

plt.tight_layout()

* Из графиков распределения видно:
    * пятница и суббота являются днями с наибольшой посещаемостью, воскресенье и понедельник - наименьшая посещаемость
    * доли вернувшихся пользователей незначительно отличаются от дня недели, что говорит об отсутствии зависимости  

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
# Создаем группы пользователей по количеству заказов
df_user_pr['order_group'] = pd.cut(df_user_pr['orders_count'],
                                    bins=[1, 2, 5, float('inf')],
                                    labels=['1 заказ', '2-4 заказа', '5 и более заказов'],
                                    include_lowest=True, right=False)

In [ ]:
# Рассчитываем среднее время между заказами для двух групп пользователей
avg_intervals = (df_user_pr.query("order_group != '1 заказ'")
                                            .groupby('order_group', observed=True)['avg_days_since_prev']
                                            .mean().astype(float).round(2))

display(f'Среднее время между заказами для групп пользователей')
display(avg_intervals)

In [ ]:
# Строим график распределения среднего времени между заказами для двух груп пользователей
avg_intervals.plot.bar(title='Распределение среднего времени между заказами для двух груп пользователей',
                       rot=0, figsize=(12,8))

# Задаем настройки графика
plt.xlabel('Группа пользователя')
plt.ylabel('Среднее время между заказами')
plt.grid(axis='y')

* Из распределения видно, что пользователи, совершившие 5 и более заказов, имеют среднее время между заказами более чем вдвое меньше, чем пользователи, совершившие 2-4 заказа.

In [ ]:
# Определяем группы пользователей
two_four_orders = df_user_pr.loc[(df_user_pr['order_group'] == '2-4 заказа')]
five_more_orders = df_user_pr.loc[(df_user_pr['order_group'] == '5 и более заказов')]

In [ ]:
# Задаем размер графика
plt.figure(figsize=(12,8))

# Строим график распределения среднего количества дней между заказами для пользователей, совершивших 2-4 заказа
sns.histplot(data=two_four_orders, x='avg_days_since_prev', bins=30, alpha=0.5, stat='density', label='2-4 заказа')
# Строим график распределения среднего количества дней между заказами для пользователей, совершивших 5 и более заказов
sns.histplot(data=five_more_orders, x='avg_days_since_prev', bins=30, color='red', stat='density', label='5 и более заказов')

# Задаем настройки графика
plt.title('Распределение среднего количества дней между заказами')
plt.xlabel('Среднее количество дней между заказами')
plt.ylabel('Плотность')
plt.legend(title='Группы пользователей')

* По графику распределения заключаем:
    * распределение среднего интервала между покупками в группе с 5 и более заказами значительно компактнее, что говорит о высокой активности покупателей
    * пользователи, совершившие 2-4 заказа, имеют сильный разброс среднего интервала между заказами, а также скос данных влево

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
# Определяем корреляцию между признаками профиля и числом заказов
columns = ['first_order', 'last_order', 'region', 'first_event_type', 'device',
           'service', 'orders_count', 'avg_revenue', 'avg_tickets_count', 'avg_days_since_prev']

correlation_matrix = (df_user_pr[columns].
                      phik_matrix(interval_cols=['orders_count', 'avg_revenue', 'avg_tickets_count', 'avg_days_since_prev'])
                      .loc['orders_count'].drop('orders_count').sort_values(ascending=False).to_frame().round(2))


display(correlation_matrix)

In [ ]:
# Строим тепловую карту
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, linewidths=0.5)

# Задаем настройки графика
plt.title('Корреляция количества заказов от признаков профиля пользователя')

* Из тепловой карты корреляции кол-ва заказов от признаков профиля пользователя видно:
    * наибольшую корреляцию показали даты первого и последнего заказаков (0.43 и 0.42 соответственно)
    * среднее кол-во купленных билетов и средний интервал между покупками имеют слабую корреляцию (0.34 и 0.3 соответственно) 
    * средняя выручка и регион, в котором была совершена первая покупка, имеют очень слабую корреляцию (0.22 и 0.12 соответственно)
    * тип устройства, тип первого мероприятия и билетный оператор имеют почти нулевую коррелциюю (0.036, 0.03 и 0.026 соответственно)

In [ ]:
# Добавим три бинарных признака по количеству заказов пользоветелей
df_user_pr = df_user_pr.assign(
                  one_order = lambda x: (x['order_group'] == '1 заказ').astype(int),
                  two_four_order = lambda x: (x['order_group'] == '2-4 заказа').astype(int),
                  five_more_order = lambda x: (x['order_group'] == '5 и более заказов').astype(int))

In [ ]:
# Определяем корреляцию для пользователей с 1 заказом
columns = ['first_order', 'last_order', 'region', 'first_event_type', 'device', 'service', 
           'avg_revenue', 'avg_days_since_prev', 'one_order']

correlation_one_order = (df_user_pr[columns].
                      phik_matrix(interval_cols=['avg_revenue', 'avg_tickets_count', 'avg_days_since_prev', 'one_order'])
                      .loc['one_order'].drop('one_order').sort_values(ascending=False).to_frame().round(2))


display(correlation_one_order)

In [ ]:
# Определяем корреляцию для пользователей с 2-4 заказами
columns = ['first_order', 'last_order', 'region', 'first_event_type', 'device', 'service', 
           'avg_revenue', 'avg_days_since_prev', 'two_four_order']

correlation_two_four_order = (df_user_pr[columns].
                      phik_matrix(interval_cols=['avg_revenue', 'avg_tickets_count', 'avg_days_since_prev', 'one_order'])
                      .loc['two_four_order'].drop('two_four_order').sort_values(ascending=False).to_frame().round(2))


display(correlation_two_four_order)

In [ ]:
# Определяем корреляцию для пользователей с 5 и более заказами
columns = ['first_order', 'last_order', 'region', 'first_event_type', 'device', 'service', 
           'avg_revenue', 'avg_days_since_prev', 'five_more_order']

correlation_five_more_order = (df_user_pr[columns].
                      phik_matrix(interval_cols=['avg_revenue', 'avg_tickets_count', 'avg_days_since_prev', 'one_order'])
                      .loc['five_more_order'].drop('five_more_order').sort_values(ascending=False).to_frame().round(2))


display(correlation_five_more_order)

In [ ]:
# Создаем сетку для графиков
fig,axes = plt.subplots(nrows=1, ncols=3, figsize=(14,6))

# Строим тепловую карту для пользователей с 1 заказом
sns.heatmap(correlation_one_order, ax=axes[0], annot=True, linewidths=0.5)

# Строим тепловую карту для пользователей с 2-4 заказами
sns.heatmap(correlation_two_four_order, ax=axes[1], annot=True, linewidths=0.5)

# Строим тепловую карту для пользователей с 5 и более заказами
sns.heatmap(correlation_five_more_order, ax=axes[2], annot=True, linewidths=0.5)

plt.tight_layout()

* Для каждого сегмента пользователей выделяются отдельные признаки.
* Для группы пользователей с 1 заказом:
    * умеренную корреляцию показали даты первого и последнго заказов, а также средняя выручка за заказ (0.45, 0.41 и 0.31 соответсвтенно)
    * остальные признаки показали почти нулевую корреляцию (от 0.03 до 0.08)
* Для группы пользователей с 2-4 заказами:
    * умеренную корреляцию показал средний интервал между заказами (0.46), даты последнго и первого заказов показали очень слабую корреляцию (0.19 и 0.17)
    * остальные показатели имеют почти нулевую корреляцию (от 0 до 0.06)
* Для группы пользователей с 5 и более заказами:
    * умеренную корреляцию показали даты последнго и первого заказов (0.57 и 0.54), также средний интервал между заказами (0.47)
    * средняя выручка имеет слабую корреляцию (0.34)
    * остальные признаки показали почти нулевую корреляцию (от 0.03 до 0.09)

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

# Информация о данных
* Работа производилась с датасетом, который включает в себя данные о 290611 заказах
    * для части данный значения выручки были представлены в казахтанских тенге, была проведена конвертация в российские рубли с учетом курса
    * было удалено 44 дубликата
    * также была проведена фильтрация данных по 99-му процентилю средней выручки за заказ для отсечения выбросов, было отсечено 3386 строк
* Были сформированы профили пользователей, всего 21831 пользователей
    * были добавлены бинарные признаки: совершено 2 и более заказов, совершено 5 и более заказов
    * была проведена фильтрация данных по 99-му процентилю кол-ва заказов для отсечения выбросов, отсечено 219 строк

# Результаты анализа
* Распределение пользователей по признакам, наиболее популярными являются:
    * Тип мероприятия: концерты
    * Тип устройства: мобильные телефоны
    * Регион проведения мероприятия: Каменевский регион
    * Билетный оператор: "Билеты без проблем"
* Влияние первого заказа на лояльность пользователей:
    * Тип мероприятия: наибольшие доли вернувшихся пользователейвыставки и театр - (64.2% и 63.4%), наименьшие - спорт и елки (55.8%)
    * Тип устройства: desktop - 63.84%, мобильные телефоны - 60.78%
    * Регион: Шанырский регион - 67.33%, Озернинский край - 55.26%
    * Билетный оператор: "Край билетов" - 65.2%, "Билеты без проблем" - 60.31%
* Поведение пользователей в зависимости от количества заказов:
    * Пользователи, совершившие один заказ: среднее значение - 545 рублей, медиана - 378 рублей, стандартное отклонение - 519 рублей, большая вариативность данных, основной диапазон средней выручки от 0 до 250 рублей
    * Пользователи, совершившие 2 и более заказов: среднее значение - 544 рублей, медиана - 497 рублей, стандартное отклонение - 399 рублей, умеренная вариативность данных, основной диапазон средней выручки от 300 до 800 рублей
    * Пользователи, совершившие 2-4 заказа: среднее значение - 551 рублей, медиана - 472 рубля, стандартное отклонение - 420 рублей, умеренная вариативность данных, основной диапазон средней выручки от 0 до 350 рублей
    * Пользователи, совершившие 5 и более заказов: среднее значение - 536 рублей, медиана - 512 рублей, стандартное отклонение - 298 рублей, компактное распределение данных, основной диапазон средней выручки от 0 до 350 рублей
* Распределение пользователей от количества билетов в заказе:
    * Распределение неравномерное, основная масса покупок приходится на 2-3 билета и на 3-5 билетов
    * Наибольшая доля вернувшихся пользователей лежит в сегменте 2-3 билетов
    * Сегменты 3-5 билетов и 1-2 билета имеют незначительное отличие в доли вернувшихся пользователей
    * Сегмент с 5 и более билетами имеет самые низкие показатели
* Влияние временных характеристик на поведение пользователей:
    * День недели: пятница и суббота - наибольшая посещаемость, а воскресенье и понедельник - наименьшая посещаемость, доли вернувшихся пользователей незначительно отличаются от дня недели
    * Время между покупками: пользователи, совершившие 5 и более заказов, имеют средний интервал между заказами более чем вдвое меньше, чем пользователи, совершившие 2-4 заказа. Распределение среднего интервала между покупками в группе с 5 и более заказами значительно компактнее, что говорит о высокой активности покупателей, пользователи, совершившие 2-4 заказа, имеют сильный разброс среднего интервала между заказами
* Корреляция признаков профиля пользователей на количество покупок:
    * Для пользователей с 1 заказом: умеренную корреляцию показали даты первого и последнго заказов, а также средняя выручка за заказ (0.45, 0.41 и 0.31 соответсвтенно)
    * Для пользователей с 2-4 заказами: умеренную корреляцию показал средний интервал между заказами (0.46), даты последнго и первого заказов показали очень слабую корреляцию (0.19 и 0.17)
    * Для группы пользователей с 5 и более заказами: умеренную корреляцию показали даты последнго и первого заказов (0.57 и 0.54), также средний интервал между заказами (0.47), средняя выручка имеет слабую корреляцию (0.34)

# Рекомендации
* Сфокусироваться на удержании пользователей, совершивших 2–3 покупки.
* Внедрить рассылку (email, push) для пользователей группы "2–4 заказа", так как у них наблюдается сильный разброс в интервалах между покупками.
* Выяснить причину низкой лояльности пользователей с 5 и более билетами в заказе. Вероятно, это корпоративные покупки или большие компании друзей. Разработать специальные условия для повышения повторных покупок.
* Увеличить доли мероприятий с высокой лояльностью (выставки, театры, конспекты), так как они приносят наиболее стабильную аудиторию.
* Для спортивных мероприятий разработать специальные программы лояльности, чтобы поднять долю вернувшихся пользователей.
* Самый популярный тип устройства — мобильные телефоны, однако доля вернувшихся пользователей там чуть ниже, чем на Desktop. Возможно, присутствуют проблемы с мобильной версией сайта или приложением.
* Разработать промо-акции для повышения посещаемости в воскресенье и понедельник. 

### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

https://github.com/AlexandrNkE/Yandex-Afisha-analysis